# 📑 Table of Contents

1. Import Required Libraries
2. Load the MedQuAD Dataset
3. Exploratory Data Analysis
4. Data Cleaning and Preprocessing
5. Train–Validation–Test Split
6. Creating LangChain Documents
7. Generating Sentence Embeddings
8. Building the FAISS Vector Database
9. Loading the TinyLlama Language Model
10. Helper Functions
11. Testing the RAG Pipeline
12. Model Evaluation
13. Gradio Web Interface
14. Testing the Chatbot
15. Conclusion
16. Future Improvements

# 🩺 Medical RAG Assistant using TinyLlama and FAISS

### Retrieval-Augmented Generation for Medical Question Answering

This notebook implements a Retrieval-Augmented Generation (RAG) based medical chatbot using the MedQuAD dataset, Sentence Transformers, FAISS, TinyLlama, and Gradio.

In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd
import re# data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gpreda/medquad/medquad.csv


# 1. Import Required Libraries

In [8]:
!pip install -q \
langchain \
langchain-community \
transformers \
sentence-transformers \
faiss-cpu \
accelerate \
bitsandbytes \
pandas \
torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60

In [9]:
import re
import pandas as pd
import torch

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

/tmp/ipykernel_58/396103186.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [10]:
import faiss
import langchain
import transformers
import sentence_transformers
import torch

# 2. Load the MedQuAD Dataset

In [11]:
df = pd.read_csv("/kaggle/input/datasets/gpreda/medquad/medquad.csv")

print(df.shape)

df.head()

(16412, 4)


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


# 3. Exploratory Data Analysis (EDA)

- Dataset Shape
- Missing Values
- Duplicate Records
- Answer Length Distribution

# 4. Data Cleaning and Preprocessing

- Cleaning Questions
- Cleaning Answers
- Removing Webpage Artifacts
- Final Dataset

In [12]:
df = df.drop(columns=["source"])

df = df.drop_duplicates(
    subset=["question","answer"]
)

df = df.dropna(
    subset=["question","answer"]
)

df["focus_area"] = df["focus_area"].fillna("Unknown")

In [13]:
def clean_question(q):

    q = str(q)

    q = re.sub(r"\(are\)", "", q)
    q = re.sub(r"\(is\)", "", q)

    q = re.sub(r"\s+"," ",q)

    q = q.replace(" ?","?")

    return q.strip()

In [51]:
def clean_answer(text):

    text = str(text)

    patterns = [

        r"\(Watch the video.*?\)",

        r"Watch the video.*",

        r"To enlarge the video.*",

        r"To reduce the video.*",

        r"See this graphic.*",

        r"See a glossary.*",

        r"Click here.*",

        r"Print this page.*",

        r"For more information.*",

        r"References.*",

        r"Additional Information.*",

        r"Related Topics.*",

        r"Learn More.*",
        r'button on your keyboard\.?'

    ]

    for p in patterns:

        text = re.sub(
            p,
            "",
            text,
            flags=re.IGNORECASE|re.DOTALL
        )

    text = re.sub(r"\s+"," ",text)

    return text.strip()

In [52]:
df["question"] = df["question"].apply(clean_question)

df["answer"] = df["answer"].apply(clean_answer)

df = df[df["answer"].str.strip()!=""]

In [68]:
import re

def clean_answer(text):
    text = str(text)

    # Cut off at any UI/navigation instructions
    cut_patterns = [
        r'button on your keyboard.*',
        r'Click.*',
        r'Press.*',
        r'Watch the video.*',
        r'See the animation.*',
        r'Written and spoken.*',
        r'Use the buttons.*',
        r'To enlarge the video.*',
        r'To reduce the video.*',
        r'Return to top.*',
        r'Print this page.*',
        r'Share this page.*'
    ]

    for pattern in cut_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [69]:
train_df["answer"] = train_df["answer"].apply(clean_answer)
val_df["answer"] = val_df["answer"].apply(clean_answer)
test_df["answer"] = test_df["answer"].apply(clean_answer)

# 5. Train–Validation–Test Split

In [38]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: 10% validation, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 13085
Validation: 1636
Test: 1636


In [74]:
validation_questions = val_df["question"].tolist()
validation_answers = val_df["answer"].tolist()
validation_categories = val_df["focus_area"].tolist()

print(len(validation_questions))

1636


# 6. Creating LangChain Documents

In [61]:
from langchain_core.documents import Document

documents = []

for _, row in train_df.iterrows():

    text = f"""
Medical Category: {row['focus_area']}

Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append(
        Document(
            page_content=text,
            metadata={
                "focus_area": row["focus_area"]
            }
        )
    )

print("Documents:", len(documents))

Documents: 13085


In [70]:
documents = []
for _, row in train_df.iterrows():
    documents.append(
        Document(
            page_content=f"""Medical Category: {row['focus_area']}

Question: {row['question']}

Answer: {row['answer']}""",
            metadata={"focus_area": row["focus_area"]}
        )
    )

# 7. Generating Sentence Embeddings

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64
    }
)

/tmp/ipykernel_58/2007849829.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 8. Building the FAISS Vector Database

In [71]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(documents, embedding_model)
print("FAISS Index Created Successfully!")

FAISS Index Created Successfully!


In [73]:
db.save_local("medical_faiss")
print("FAISS index saved successfully!")

FAISS index saved successfully!


In [76]:
db.save_local("medical_faiss")

# 9. Loading the TinyLlama Language Model

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,                  # CPU (or GPU if compatible)
    max_new_tokens=180,
    do_sample=False,
    temperature=0.1,
    repetition_penalty=1.15,
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'pad_token_id', 'eos_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# 10. Helper Functions

### Detect User Intent

### Medical Question Answering Function

In [30]:
def detect_intent(query):
    query = query.lower()

    if "what is" in query or "define" in query:
        return "definition"

    if "cause" in query or "why" in query:
        return "causes"

    if "symptom" in query:
        return "symptoms"

    if "treatment" in query or "treat" in query:
        return "treatment"

    if "prevent" in query:
        return "prevention"

    return "general"

In [83]:
def ask_medical_rag(query, verified_threshold=0.55, medical_threshold=1.23):

    intent = detect_intent(query)

    # Retrieve the most relevant document
    results = db.similarity_search_with_score(query, k=1)

    if not results:
        return "❌ No relevant medical information found."

    doc, score = results[0]

    confidence = max(0, min(100, (1 - score) * 100))

    # ==========================================================
    # VERIFIED RAG ANSWER
    # ==========================================================
    if score <= verified_threshold:

        page = doc.page_content

        if "Answer:" in page:
            context = page.split("Answer:", 1)[1].strip()
        else:
            context = page

        prompt = f"""
You are a medical assistant.

Use ONLY the medical information below.

The user is asking for the {intent}.

Rules:
- Answer only the user's question.
- Do not repeat the context.
- Do not output "Question:" or "Answer:".
- Keep the answer concise (3-6 sentences).
- If the information is insufficient, say:
"I don't have enough information."

Medical Information:
{context}

User Question:
{query}

Answer:
"""

        answer = pipe(
            prompt,
            max_new_tokens=180,
            do_sample=False,
            return_full_text=False
        )[0]["generated_text"].strip()

        category = doc.metadata.get("focus_area", "Unknown")

        return f"""✅ Verified Medical Answer

{answer}

────────────────────────────

📚 Category: {category}

🎯 Retrieval Confidence: {confidence:.2f}%

📖 Source: MedQuAD Knowledge Base
"""

    # ==========================================================
    # GENERAL MEDICAL FALLBACK
    # ==========================================================
    elif score <= medical_threshold:

        prompt = f"""
You are a knowledgeable medical assistant.

The user's intent is: {intent}

Answer only that aspect.

If the user asks:
- What is -> give the definition.
- Symptoms -> explain symptoms only.
- Causes -> explain causes only.
- Treatment -> explain treatment only.

Question:
{query}

Answer:
"""

        answer = pipe(
            prompt,
            max_new_tokens=180,
            do_sample=False,
            return_full_text=False
        )[0]["generated_text"].strip()

        return f"""⚠️ General Medical Knowledge

{answer}

────────────────────────────

⚠️ This topic was not found in the verified MedQuAD knowledge base.

The answer above was generated using TinyLlama's general medical knowledge and should not be considered a verified medical reference.
"""

    # ==========================================================
    # NON-MEDICAL QUERY
    # ==========================================================
    else:

        return (
            "❌ I am a medical assistant and can only answer "
            "health and medical-related questions."
        )

# 11. Testing the RAG Pipeline

In [72]:
query = "What is glaucoma?"

docs = db.similarity_search(query, k=1)

print(docs[0].page_content)

Medical Category: Glaucoma

Question: What is Glaucoma?

Answer: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. The most common form of the disease is open-angle glaucoma. With early treatment, you can often protect your eyes against serious vision loss.


In [80]:
print(ask_medical_rag("what is heart attack?"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Verified Medical Answer

a heart attack occurs when the supply of blood and oxygen to an area of the heart muscle is blocked, usually by a blood clot in a coronary artery.

────────────────────────────

📚 Category: Heart Attack

🎯 Retrieval Confidence: 66.19%

📖 Source: MedQuAD Knowledge Base



In [85]:
ask_medical_rag("What is Marburg virus disease")

Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"⚠️ General Medical Knowledge\n\nMarburg virus disease is a rare and deadly viral infection caused by Marburg virus, which belongs to the family of filoviruses. It primarily affects animals such as monkeys, but it can also infect humans. The disease is named after the city of Marburg, Germany where the first human case was reported in 1967. Symptoms include fever, headache, muscle pain, vomiting, diarrhea, and sometimes hemorrhagic manifestations (bleeding from internal organs). The mortality rate for this disease is high, with an estimated range between 20% and 85%. Early diagnosis and appropriate treatment can help improve outcomes.\n\n────────────────────────────\n\n⚠️ This topic was not found in the verified MedQuAD knowledge base.\n\nThe answer above was generated using TinyLlama's general medical knowledge and should not be considered a verified medical reference.\n"

In [84]:
print(ask_medical_rag("What is usa?"))

❌ I am a medical assistant and can only answer health and medical-related questions.


In [86]:
test_questions = [
    "What is glaucoma?",
    "What causes diabetes?",
    "What is lung cancer?",
    "What are the symptoms of stroke?",
    "What is osteoporosis?",
    "What causes Alzheimer's disease?",
    "What is psoriasis?",
    "What is leukemia?",
    "What is Parkinson's disease?",
    "What is heart failure?"
]

In [87]:
for q in test_questions:
    print("=" * 80)
    print("Question:", q)
    print()
    print(ask_medical_rag(q))
    print()

Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is glaucoma?



Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Verified Medical Answer

Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. The most common form of the disease is open-angle glaucoma. Early detection and treatment with medication or surgery can help prevent severe vision loss.

────────────────────────────

📚 Category: Glaucoma

🎯 Retrieval Confidence: 53.68%

📖 Source: MedQuAD Knowledge Base


Question: What causes diabetes?



Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

✅ Verified Medical Answer

Psoriasis is a chronic, inflammatory disease characterized by thick, itchy, silvery scales covering the skin. It affects about one percent of the population worldwide. Psoriasis can be caused by various factors such as genetics, environmental exposure, autoimmune disorders, and infections. Guttate psoriasis is a rare type of psoriasis that causes small, red, and scaly spots on the skin.

────────────────────────────

📚 Category: Guttate psoriasis

🎯 Retrieval Confidence: 49.67%

📖 Source: MedQuAD Knowledge Base


Question: What is leukemia?



Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚠️ General Medical Knowledge

Leukemia is a type of cancer that affects white blood cells, specifically red and white blood cells. It can occur in both adults and children. The most common types of leukemia include acute lymphocytic leukemia (ALL), chronic lymphocytic leukemia (CLL), and myelodysplastic syndrome (MDS). Leukemia can be treated with chemotherapy, radiation therapy, or bone marrow transplantation.

────────────────────────────

⚠️ This topic was not found in the verified MedQuAD knowledge base.

The answer above was generated using TinyLlama's general medical knowledge and should not be considered a verified medical reference.


Question: What is Parkinson's disease?



Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Verified Medical Answer

Parkinson's disease is a neurodegenerative disorder characterized by the degeneration of dopaminergic neurons in the substantia nigra pars compacta, which produces dopamine. The exact cause of Parkinson's disease is unknown but is believed to involve multiple factors including genetics, environmental exposures, and lifestyle choices such as smoking and alcohol consumption. Symptoms include tremors, rigidity, bradykinesia (slow movement), postural instability, and sleep disturbances. Parkinson's disease can lead to other complications such as depression, cognitive impairment, and falls.

────────────────────────────

📚 Category: Parkinson's Disease

🎯 Retrieval Confidence: 56.08%

📖 Source: MedQuAD Knowledge Base


Question: What is heart failure?

✅ Verified Medical Answer

Heart failure is a condition in which the heart cannot pump enough blood to meet the body's needs. It occurs when the heart muscle becomes weakened, causing the heart to be unable to pump 

# 12. Model Evaluation

### Top-1 Retrieval Accuracy

### Top-3 Retrieval Accuracy

### Precision, Recall and F1-Score

### ROUGE-L Evaluation

### BERTScore Evaluation

### Average Inference Time

In [90]:
!pip install -q rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


In [91]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

In [96]:
import time
from rouge_score import rouge_scorer
from bert_score import score as bertscore

# Evaluate on 100 random samples from the TEST set
sample_test = test_df.sample(n=100, random_state=42)

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

predictions = []
references = []
times = []
rouge_scores = []

for _, row in sample_test.iterrows():

    question = row["question"]
    reference = row["answer"]

    start = time.time()

    prediction = ask_medical_rag(question)

    end = time.time()

    predictions.append(prediction)
    references.append(reference)
    times.append(end - start)

    rouge_scores.append(
        rouge.score(reference, prediction)["rougeL"].fmeasure
    )

# Compute BERTScore ONCE (much faster)
P, R, F1 = bertscore(
    predictions,
    references,
    lang="en",
    device="cpu",      # Keep CPU because of your CUDA compatibility issue
    verbose=False
)

print(f"Evaluation Samples      : {len(sample_test)}")
print(f"Average ROUGE-L         : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERTScore (F1)  : {F1.mean().item():.4f}")
print(f"Average Inference Time  : {sum(times)/len(times):.4f} seconds")

Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluation Samples      : 100
Average ROUGE-L         : 0.1597
Average BERTScore (F1)  : 0.8237
Average Inference Time  : 24.6540 seconds


In [97]:
correct = 0
total = len(val_df)

for _, row in val_df.iterrows():

    query = row["question"]
    expected = row["focus_area"]

    docs = db.similarity_search(query, k=3)

    predicted_categories = [doc.metadata["focus_area"] for doc in docs]

    if expected in predicted_categories:
        correct += 1

top3_accuracy = correct / total

print(f"Top-3 Retrieval Accuracy: {top3_accuracy:.4f}")

Top-3 Retrieval Accuracy: 0.6962


In [98]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []

for _, row in val_df.iterrows():

    docs = db.similarity_search(row["question"], k=1)

    expected = row["focus_area"]
    predicted = docs[0].metadata["focus_area"]

    y_true.append(expected)
    y_pred.append(predicted)

precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

Precision : 0.4942
Recall    : 0.5043
F1 Score  : 0.4872


# 13. Gradio Web Interface

In [100]:
!pip install -q gradio

In [102]:
def chatbot(question):
    try:
        return ask_medical_rag(question)
    except Exception as e:
        return f"Error: {e}"

In [103]:
import gradio as gr

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a medical question...",
        label="Medical Question"
    ),
    outputs=gr.Textbox(
        label="Answer"
    ),
    title="🩺 Medical RAG Chatbot",
    description="Medical Question Answering using TinyLlama + FAISS + MiniLM",
    theme=gr.themes.Soft()
)

In [104]:
import gradio as gr

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Example: What are the symptoms of glaucoma?",
        label="🩺 Enter your medical question"
    ),
    outputs=gr.Textbox(
        lines=10,
        label="📋 Medical Answer"
    ),
    title="# 🏥 Medical RAG Assistant",
    description="""
A Retrieval-Augmented Generation (RAG) chatbot built using:

• TinyLlama-1.1B-Chat
• FAISS Vector Database
• Sentence Transformers (MiniLM)
• MedQuAD Dataset

⚠️ This chatbot is for educational purposes only and should not replace professional medical advice.
""",
    allow_flagging="never",
    theme=gr.themes.Soft(),
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


# 14. Testing the Chatbot

In [105]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b78d66624deab1cfc9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [106]:
demo.close()

Closing server running on port: 7860


# 15. Conclusion

### Key Features

- Retrieval-Augmented Generation (RAG)
- Semantic Search using FAISS
- TinyLlama Response Generation
- Confidence-based Retrieval
- Interactive Gradio Interface

In [77]:
requirements = """
torch
transformers
sentence-transformers
langchain
langchain-community
langchain-core
faiss-cpu
gradio
pandas
numpy
scikit-learn
bert-score
rouge-score
accelerate
"""

with open("requirements.txt","w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [99]:
import os

os.makedirs("saved_rag", exist_ok=True)

db.save_local("saved_rag/faiss_index")

tokenizer.save_pretrained("saved_rag/tinyllama")

model.save_pretrained("saved_rag/tinyllama")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# 16. Future Improvements

- BioBERT Embeddings
- PubMed Integration
- Medical Reranking
- Better LLM (Phi-3 / Llama 3)
- Cloud Deployment